In [3]:
from  openai import OpenAI
from qdrant_client import QdrantClient

### Embedding fuction

In [4]:
client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",  
)

In [5]:
def get_embedding(text, model="nomic-embed-text"):
    response = client.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

### Retrieval function

In [6]:
qdrant_client = QdrantClient(url="http://localhost:6333")

In [11]:
def retrieve_data(query , qdrant_client, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="Amazon_items_collection1",
        query=query_embedding,
        limit=top_k
    )

    retrieved_context_ids = []
    retrieved_contexts = []
    similarity_scores = []
    retrieved_context_ratings = []

    for point in search_result.points:
        retrieved_context_ids.append(point.payload["parent_asin"])
        retrieved_contexts.append(point.payload["description"])
        retrieved_context_ratings.append(point.payload["average_rating"])
        similarity_scores.append(point.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_contexts": retrieved_contexts,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

In [12]:
retrieved_data = retrieve_data("What are some good electronics items with high ratings?", qdrant_client, top_k=10)

In [13]:
retrieved_data

{'retrieved_context_ids': ['B0BDM7F3CT',
  'B0BNQ951N8',
  'B0BNQ951N8',
  'B09QHPCTHG',
  'B09JV5FM2S',
  'B0C6ZDBTXT',
  'B0BTBTPSZL',
  'B0BQJ9MFD3',
  'B0B715Y19L',
  'B08Q27PMFK'],
 'retrieved_contexts': ['uni USB C to USB C Video Cable 8K, USB4 C Monitor Cable Supports 4K Display/40Gbps Speedy Data Transfer/100W Charging Compatible with Thunderbolt 3/4 MacBook, iPad Pro, Samsung, HP 【8K Crystal-Clear Visual】uni USB-C to USB-C Video Cable for display output supports 8K/5K@60Hz, 4K@144Hz UHD single monitor at 60Hz or dual 4K monitors at 60Hz through a daisy chain. Free to enjoy stunning films with vivid colors on a bigger TV. Plus, capable of playing an immersive game and elevates your game for a pro-level experience. 【Surprising High Refresh Rate】uni USB Type C Display Cable is your first choice for gaming. The Ultra-High refresh rate will improve your breath-taking gaming performance. See your enemies before they see you. Note: The actual resolution depends on your devices. And m

### Fromat retrieved context function

In [14]:
def process_context(context):
    processed_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_contexts"], context["retrieved_context_ratings"]):
        processed_context += f"-ID {id}: rating: {rating}, description: {chunk}\n"
    return processed_context

In [15]:
processed_context = process_context(retrieved_data)
print(processed_context)

-ID B0BDM7F3CT: rating: 4.6, description: uni USB C to USB C Video Cable 8K, USB4 C Monitor Cable Supports 4K Display/40Gbps Speedy Data Transfer/100W Charging Compatible with Thunderbolt 3/4 MacBook, iPad Pro, Samsung, HP 【8K Crystal-Clear Visual】uni USB-C to USB-C Video Cable for display output supports 8K/5K@60Hz, 4K@144Hz UHD single monitor at 60Hz or dual 4K monitors at 60Hz through a daisy chain. Free to enjoy stunning films with vivid colors on a bigger TV. Plus, capable of playing an immersive game and elevates your game for a pro-level experience. 【Surprising High Refresh Rate】uni USB Type C Display Cable is your first choice for gaming. The Ultra-High refresh rate will improve your breath-taking gaming performance. See your enemies before they see you. Note: The actual resolution depends on your devices. And make sure your USB-C devices support Alt DisplayPort (DP Alt Mode). 【Faster Life with 40Gbps Transfer】Achieves a high data transfer rate UP to 40Gbps / 5000MB/s and a ful

#### Create prompt function

In [16]:
def build_prompt(query, context):
    prompt = f"""You are shopping assistant that can answer questions about products. 
    You will be given a question and some retrieved information about products.
    
    Instructions:
    - You need to use only the retrieved information to answer the question.
    - Never use word context and refer to it as the available products.

    Context:
    {context}

    Question:
    {query}  """

    return prompt

In [17]:
prompt = build_prompt("What are some good electronics items with high ratings?", processed_context)
print(prompt)

You are shopping assistant that can answer questions about products. 
    You will be given a question and some retrieved information about products.

    Instructions:
    - You need to use only the retrieved information to answer the question.
    - Never use word context and refer to it as the available products.

    Context:
    -ID B0BDM7F3CT: rating: 4.6, description: uni USB C to USB C Video Cable 8K, USB4 C Monitor Cable Supports 4K Display/40Gbps Speedy Data Transfer/100W Charging Compatible with Thunderbolt 3/4 MacBook, iPad Pro, Samsung, HP 【8K Crystal-Clear Visual】uni USB-C to USB-C Video Cable for display output supports 8K/5K@60Hz, 4K@144Hz UHD single monitor at 60Hz or dual 4K monitors at 60Hz through a daisy chain. Free to enjoy stunning films with vivid colors on a bigger TV. Plus, capable of playing an immersive game and elevates your game for a pro-level experience. 【Surprising High Refresh Rate】uni USB Type C Display Cable is your first choice for gaming. The Ultra

#### Generate answer function

In [18]:
def generate_answer(prompt):
    response = client.chat.completions.create(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": "You are a helpful shopping assistant that can answer questions about products based on the available products."},
            {"role": "user", "content": prompt}
        ],
    )
    return response.choices[0].message.content

In [19]:
print(generate_answer(prompt))

Okay, let's identify some of the electronics items with high ratings from the provided descriptions. Based on the ratings given (generally higher is better), here’s a breakdown:

**Top Tier (5-star ratings):**

*   **PNY 256GB PRO Elite Class 10 microSDXC Flash Memory Card + SanDisk MobileMate Reader:** This combo consistently receives a 4.5-star rating, indicating very positive feedback. People praise its speed (100MB/s read), compatibility, and the included reader.

*   **EGQINR Smart Watch:** This watch has a 4.2-star rating, which is quite high. People appreciate the blood pressure monitoring, call functionality, and overall features.

*   **MIPEACE Bluetooth Work Earplugs Headphone:** This gets a 4.2-star rating, highlighting its effectiveness in noise reduction, comfortable design, and battery life.

**Good Tier (4-star ratings):**

*   **JOSTART 【4 Pack】 USB C to USB Adapter:** This adapter receives a 4.4-star rating. The quality of its construction for its purpose is appreciate

#### Combined RAG pipeline


In [20]:
def rag_pipeline(query, top_k=5):
    retrieved_data = retrieve_data(query, qdrant_client, top_k)
    processed_context = process_context(retrieved_data)
    prompt = build_prompt(query, processed_context)
    answer = generate_answer(prompt)
    return answer

In [21]:
print(rag_pipeline("What are some good electronics items with high ratings?"))

Here are the electronics items with high ratings based on the available products:

-ID B0BDM7F3CT: rating: 4.6
-ID B0BNQ951N8: rating: 4.2 (appears twice)
-ID B09QHPCTHG: rating: 4.4
-ID B09JV5FM2S: rating: 4.5
